# 2.4 PromptTemplate Deep Dive with LangChain

## 🎯 Learning Objectives

This notebook provides a **comprehensive exploration** of LangChain's prompt templating system:

1. **PromptTemplate** - For text completion LLMs
2. **ChatPromptTemplate** - For chat models with message roles
3. **format() vs format_prompt() vs invoke()** - Understanding the differences
4. **LCEL Integration** - Using prompts in chains

## 🔑 Quick Reference

| Method | Returns | Use Case |
|--------|---------|----------|
| `.format()` | `str` | Simple string formatting |
| `.format_prompt()` | `PromptValue` | When you need the intermediate object |
| `.invoke()` | `PromptValue` | For LCEL chains |

---

In [ ]:
# ============================================================================
# INSTALLATION (Uncomment if needed)
# ============================================================================
# !pip install -qq langchain
# !pip install -qq langchain-openai
# !pip install -qq langchain-community
# !pip install -qq openai

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

print("✅ Core imports loaded")

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================

import os
import openai
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']
print("✅ API keys loaded")

---

## 📝 Part 1: Basic PromptTemplate

`PromptTemplate` is used for creating **parameterized prompts** with placeholders.

In [ ]:
# ============================================================================
# CREATING A PROMPTTEMPLATE
# ============================================================================
# Two ways to create:
# 1. Using the constructor with input_variables (explicit validation)
# 2. Using .from_template() (auto-detects variables)
# ============================================================================

template = """You are an expert in deep learning and PyTorch.
You answer queries by being brief, bright, and concise.

Query: {query}
"""

# Method 1: Explicit input_variables (recommended for production)
prompt_template = PromptTemplate(input_variables=['query'], template=template)
print("📋 Method 1 - Explicit variables:")
prompt_template.pretty_print()

# Method 2: Auto-detect from template (quick prototyping)
prompt_template = PromptTemplate.from_template(template)
print("\n📋 Method 2 - Auto-detected variables:")
prompt_template.pretty_print()

In [ ]:
# ============================================================================
# .format() - Returns a String
# ============================================================================
# Use when you need the final prompt as a plain string
# ============================================================================

formatted_prompt = prompt_template.format(query="What is the best way to learn PyTorch?")

print("📝 Formatted prompt:")
print(formatted_prompt)
print(f"\n📊 Type: {type(formatted_prompt)}")  # <class 'str'>


In [ ]:
# ============================================================================
# .format_prompt() - Returns a PromptValue
# ============================================================================
# Use when you need the intermediate PromptValue object
# PromptValue can be converted to string or messages as needed
# ============================================================================

formatted_prompt_value = prompt_template.format_prompt(query="What is the best way to learn PyTorch?")

print("📝 PromptValue object:")
print(formatted_prompt_value)
print(f"\n📊 Type: {type(formatted_prompt_value)}")  # StringPromptValue

### Sending Prompts to the LLM

Both string prompts and PromptValue objects can be passed directly to `.invoke()`

In [ ]:
# ============================================================================
# INVOKING LLM WITH STRING PROMPT
# ============================================================================

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.0)

# Pass the string prompt directly
response = llm.invoke(formatted_prompt)
print("🤖 Response (from string prompt):")
print(response.content)

In [ ]:
# Pass PromptValue - works the same way!
response = llm.invoke(formatted_prompt_value)
print("🤖 Response (from PromptValue):")
print(response.content)

In [ ]:
# Response is always an AIMessage
print(f"📊 Response type: {type(response)}")

### Using Prompts in LCEL Chains

The **LangChain Expression Language (LCEL)** uses the `|` operator to chain components.

In [ ]:
# ============================================================================
# LCEL CHAIN: prompt → LLM → output parser
# ============================================================================
# The pipe operator (|) connects components
# Each component's output becomes the next component's input
# ============================================================================

chain = prompt_template | llm | StrOutputParser()

# Now we pass a dict with the template variables
response = chain.invoke({"query": "What is the best way to learn PyTorch?"})

print("🔗 LCEL Chain Response:")
print(response)

---

## 📝 Part 2: Multi-Input Prompts

Templates can have **multiple variables** for more dynamic prompts.

In [ ]:
# ============================================================================
# MULTI-INPUT PROMPT EXAMPLE
# ============================================================================
# This function demonstrates using a prompt with multiple variables
# ============================================================================

def get_movie_information(movie_title: str, main_actor: str) -> str:
    """
    Generate a fictitious movie synopsis using an LLM.
    
    Args:
        movie_title: The title of the movie
        main_actor: The main actor of the movie
    Returns:
        Generated synopsis and genre
    """
    prompt = PromptTemplate(
        template="""Your task is to create a fictitious movie synopsis and genre for the following movie and main actor:
        Movie: {movie_title}
        Actor: {main_actor}""",
        input_variables=['movie_title', 'main_actor']
    )
    
    formatted_prompt = prompt.format(movie_title=movie_title, main_actor=main_actor)
    response = llm.invoke(formatted_prompt)
    
    return response.content

# Test the function
movie_info = get_movie_information("The Dark Knight", "Christian Bale")
print("🎬 Generated Movie Info:")
print(movie_info)

### Same Function Using LCEL (with Streaming)

In [ ]:
# ============================================================================
# LCEL VERSION WITH STREAMING
# ============================================================================
# .stream() provides token-by-token output for better UX
# ============================================================================

def get_movie_information_streaming(movie_title: str, main_actor: str) -> str:
    """Generate movie info with streaming output."""
    
    prompt = PromptTemplate.from_template(template="""
        Your task is to create a fictitious movie synopsis and genre for:
        Movie: {movie_title}
        Actor: {main_actor}
        """)

    llm_chain = prompt | llm | StrOutputParser()

    # Stream output token by token
    print("🎬 Streaming response:")
    for chunk in llm_chain.stream({"movie_title": movie_title, "main_actor": main_actor}):
        print(chunk, end="", flush=True)
    
    print("\n")  # New line after streaming
    
    # Return the complete response
    return llm_chain.invoke({"movie_title": movie_title, "main_actor": main_actor})

# Test with streaming
result = get_movie_information_streaming(movie_title="Amritsar:1984", main_actor="Gurdaas Mann")

---

## 🧱 Part 3: PromptTemplate vs ChatPromptTemplate

### PromptTemplate (for Text Completion LLMs)

- **What it does:** Builds a single formatted string prompt
- **When to use:** Text completion models (gpt-3.5-turbo-instruct, text-davinci)
- **Output:** `StringPromptValue`

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define a prompt with variables
prompt = PromptTemplate.from_template("Translate this sentence to French: {sentence}")

# Format the prompt
formatted_prompt = prompt.format(sentence="I love programming.")
print(formatted_prompt)

In [ ]:
fmt_prompt = prompt.invoke({"sentence":"What is the best way to learn PyTorch?"})
response = llm.invoke(fmt_prompt)
print(response.content)

## **💬 2. ChatPromptTemplate (for Chat Models like GPT-3.5, GPT-4)**

### **✅ What it does:**

Creates **multi-turn prompts** in a chat format using **roles** like system, user, and ai.

### **✅ When to use:**

When using **OpenAI’s chat models**, like GPT-3.5/4 (gpt-4), **Anthropic Claude**, etc.

In [ ]:
# ============================================================================
# CHATPROMPTTEMPLATE EXAMPLE
# ============================================================================
# Use tuples of (role, content) to define messages
# Roles: "system", "user" (or "human"), "ai" (or "assistant")
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates text to French."),
    ("user", "Translate this sentence: {sentence}")
])

# .format() returns a string representation
formatted_chat_prompt = chat_prompt.format(sentence="I love programming.")
print("📝 Formatted (string):")
print(formatted_chat_prompt)


In [ ]:
type(formatted_chat_prompt)

In [ ]:
# Format the chat prompt
formatted_chat_prompt = chat_prompt.invoke({"sentence":"I love programming."})
for msg in formatted_chat_prompt.messages:
    print(msg)

In [ ]:
formatted_chat_prompt.messages

In [ ]:
formatted_chat_prompt = chat_prompt.format_prompt(sentence="I love programming.")
type(formatted_chat_prompt)


### Another Example of ChatPromptTemplate

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, yet slightly quirky and cheeky AI bot. Your name is {name}."),
    ("human", "Yo! Wassup nephew."),
    ("ai", "As an AI language model, I am incapable of being your nephew."),
    ("human", "{user_input}"),
])

In [ ]:
type(template)
# langchain_core.prompts.chat.ChatPromptTemplate

In [ ]:
template.input_variables


In [ ]:
print(template)

In [ ]:
template.messages

In [ ]:
messages = template.format_messages(
    name="Robotalker",
    user_input="Talk robo to me!"
)
print(messages)

# [SystemMessage(content='You are a helpful, yet slightly quirky and cheeky AI bot. Your name is Robotalker.'),
# HumanMessage(content='Yo! Wassup nephew.'),
# AIMessage(content='As an AI language model, I am incapable of being your nephew.'),
#HumanMessage(content='Talk robo to me!')]

print(llm.invoke(messages).content)
# Beep boop! Let's chat about all things robotic and techy. Got any burning questions about robots or artificial intelligence?

# use LCEL
chain = template | llm | StrOutputParser()

chain.invoke({"name":"Robotalker","user_input":"Talk robo to me!"})
# "Beep boop! What's shakin', human friend?"

In [ ]:
from langchain_core.prompts import PromptTemplate,HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

In [ ]:
# Another way to create Template using SystemMessage and HumanMessagePromptTemplate

system_message = SystemMessage(content="You are an OG language model who has good heart (operating system) but a bad user interface (you're super freaking rude).")
human_message = HumanMessagePromptTemplate.from_template("{text}")
template = ChatPromptTemplate.from_messages([system_message, human_message])
print(template)

In [ ]:
template.input_variables
# ['text']

In [ ]:
template.messages

In [ ]:
response = llm.invoke(template.format_messages(text="That Sam I Am, I do not like that Sam I Am..."))

print(response.content)

In [ ]:
chain = template | llm | StrOutputParser()
chain.invoke({"text":"That Sam I Am, I do not like that Sam I Am..."})

---

## 🔄 Part 4: PromptTemplate vs PromptValue

Understanding the difference is key to using LangChain effectively.

In [ ]:
# ============================================================================
# PROMPTTEMPLATE VS PROMPTVALUE
# ============================================================================
# PromptTemplate: A blueprint/factory (like a class)
# PromptValue: A specific instance with values filled in (like an object)
# ============================================================================

# Step 1: Create a PromptTemplate (the blueprint)
prompt_template = PromptTemplate.from_template("Write a joke about {topic}")

# Step 2: Use format_prompt() to create a PromptValue (specific instance)
prompt_value = prompt_template.format_prompt(topic="penguins")

print("📋 PromptTemplate (blueprint):")
print(f"   Type: {type(prompt_template)}")
print(f"   Variables: {prompt_template.input_variables}")

print("\n📦 PromptValue (instance):")
print(f"   Type: {type(prompt_value)}")
print(f"   Content: {prompt_value.to_string()}")

### **🧠 When to Use Each**

- **Use PromptTemplate**:
    - When designing **reusable and parameterized prompts**.
    - For building **dynamic chains** with variables.
- **Use PromptValue**:
    - When you want to **pass a fully formatted prompt** to an LLM manually.
    - When chaining LLMs directly: llm.invoke(prompt_value)

#### 🧪 BONUS: Use PromptValue directly with an LLM

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You're a helpful assistant."),
    ("user", "Tell me a joke about {animal}")
])

# Create a PromptValue (chat prompt formatted)
prompt_value = chat_prompt.format_prompt(animal="ducks")

# Send PromptValue to LLM directly
response = llm.invoke(prompt_value)
print(response.content)

1. Using format_prompt()

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("What is the capital of {country}?")

# Convert to PromptValue
prompt_value = template.format_prompt(country="France")
print(prompt_value.to_string())

In [ ]:
type(prompt_value)

Using format() → Return Str not a PromptValue


In [ ]:
formatted_string = template.format(country="Germany")
print(formatted_string)

In [ ]:
type(formatted_string)

For ChatPromptTemplate: .format_prompt() → returns ChatPromptValue

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You're a helpful assistant."),
    ("user", "Tell me a joke about {topic}")
])

chat_prompt_value = chat_template.format_prompt(topic="penguins")

print(chat_prompt_value.to_messages())  # List of ChatMessages

In [ ]:
type(chat_prompt_value)

In [ ]:
print(chat_prompt_value.messages)

Using Invoke()


In [ ]:
prompt = PromptTemplate.from_template("Write a poem about {topic}")

# Using invoke to get PromptValue
prompt_value = prompt.invoke({"topic": "stars"})
print(type(prompt_value))
print(prompt_value.to_string())

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You're a helpful assistant."),
    ("user", "Tell me a joke about {animal}")
])

# Using invoke
chat_prompt_value = chat_prompt.invoke({"animal": "ducks"})
print(chat_prompt_value.to_messages())

5. Use Inside LCEL Chains

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI()
parser = StrOutputParser()

chain = chat_template | llm | parser
print(chain.invoke({"topic": "cats"}))  # `format_prompt()` happens internally

## Chat Models and LLMs

Large Language Models (LLMs) are a core component of LangChain. LangChain does not implement or build its own LLMs. It provides a standard API for interacting with almost every LLM out there.

There are lots of LLM providers (OpenAI, Hugging Face, etc) - the LLM class is designed to provide a standard interface for all of them.

## Accessing Commercial LLMs like ChatGPT

In [ ]:
from langchain_openai import ChatOpenAI

# Updated parameter name from model_name to model:
chatgpt = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Prompt Templates
Prompt templates are pre-designed formats used to generate prompts for language models. These templates can include instructions, few-shot examples, and specific contexts and questions suited for particular tasks.

LangChain provides tools for creating and using prompt templates. It aims to develop model-agnostic templates to facilitate the reuse of existing templates across different language models. Typically, these models expect prompts in the form of either a string or a list of chat messages.

### Types of Prompt Templates

- **PromptTemplate:**
  - Used for creating string-based prompts.
  - Utilizes Python's `str.format` syntax for templating, supporting any number of variables, including scenarios with no variables.

- **ChatPromptTemplate:**
  - Designed for chat models, where the prompt consists of a list of chat messages.
  - Each chat message includes content and a role parameter. For instance, in the OpenAI Chat Completions API, a chat message could be assigned to an AI assistant, a human, or a system role.
- **FewShotChatMessagePromptTemplate**
  - A few-shot prompt template can be constructed from a set of examples


### PromptTemplate

We can use `PromptTemplate` to create a template for a string prompt.

By default, `PromptTemplate` uses Python's `str.format` syntax for templating.

You can create custom prompt templates that format the prompt in any way you want. For more information, see [Prompt Template Composition](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/composition/).

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = """Explain to me what is Generative AI in 3 bullet points?"""

prompt_template1 = PromptTemplate.from_template(prompt)
prompt_template = PromptTemplate(template = prompt)


In [ ]:
print(prompt_template)

In [ ]:
print(prompt_template1)

In [ ]:
# Call String as Prompt
response = chatgpt.invoke("""Explain to me what is Generative AI in 3 bullet points?""")
print(response.content)

In [ ]:
# Call PromptValue 
response = chatgpt.invoke(prompt_template1.format()) # Here there is not input variables in the prompt template. So format() does not have any input variables..
print(response.content)

In [ ]:
# more complex prompt with placeholders
prompt = """Explain to me briefly about {topic} in {language}."""

prompt_template = PromptTemplate.from_template(prompt)
prompt_template1 = PromptTemplate(template = prompt, input_variables=["topic", "language"])
prompt_value = prompt_template.format(topic="Generative AI", language="Python")
print(prompt_value)


In [ ]:
inputs = [("Generative AI", "english"),
          ("Artificial Intelligence", "hindi"),
          ("Deep Learning", "german")]

prompts = [prompt_template.format(topic=topic, language=language) for topic, language in inputs]
prompts

In [ ]:
# use map to run on multiple prompts in one go
responses = chatgpt.map().invoke(prompts)

In [ ]:
for response in responses:
  print(response.content)
  print('-----')

### ChatPromptTemplate

The standard prompt format to [chat models](https://python.langchain.com/v0.1/docs/modules/model_io/chat/) is a list of [chat messages](https://python.langchain.com/v0.1/docs/modules/model_io/chat/message_types/).

Each chat message is associated with content, and an additional parameter called `role`. For example, in the OpenAI Chat Completions API, a chat message can be associated with an AI assistant, a human or a system role.

In [ ]:
# Updated import paths for prompt templates - using simplified paths:
from langchain_core.prompts import ChatPromptTemplate

# simple prompt with placeholders
prompt = """Explain to me briefly about {topic}."""

# chat_template = ChatPromptTemplate.from_template(prompt)
chat_template = ChatPromptTemplate.from_messages([prompt])
chat_template

In [ ]:
topics = ['mortgage', 'fractional real estate', 'commercial real estate']
prompts = [chat_template.format(topic=topic) for topic in topics]
prompts

In [ ]:
responses = chatgpt.map().invoke(prompts)
for response in responses:
  print(response.content)
  print('-----')

In [ ]:
print(responses[0].content)

In [ ]:
# more complex prompt with a series of messages
messages = [
        ("system", "Act as an expert in real estate and provide brief answers"),
        ("human", "what is your name?"),
        ("ai", "my name is AIBot"),
        ("human", "{user_prompt}"),
]
chat_template = ChatPromptTemplate.from_messages(messages)
print(chat_template)

In [ ]:
text_prompts = ["what is your name?",
                "explain commercial real estate to me"]
chat_prompts = [chat_template.format(user_prompt=prompt) for prompt in text_prompts]
chat_prompts

In [ ]:
print(chat_prompts[0])

In [ ]:
responses = chatgpt.map().invoke(chat_prompts)
for response in responses:
  print(response.content)
  print('-----')

In [ ]:
messages = [
        ("system", "Act as an expert in real estate and provide very detailed answers with examples"),
        ("human", "what is your name?"),
        ("ai", "my name is AIBot"),
        ("human", "{user_prompt}"),
]
chat_template = ChatPromptTemplate.from_messages(messages)
text_prompts = ["what is your name?", "explain commercial real estate to me"]
chat_prompts = [chat_template.format(user_prompt=prompt) for prompt in text_prompts]
chat_prompts

In [ ]:
responses = chatgpt.map().invoke(chat_prompts)
for response in responses:
  print(response.content)
  print('-----')

#### PromptTemplate and ChatPromptTemplate supports LCEL

`PromptTemplate` and `ChatPromptTemplate` implement the [Runnable interface](https://python.langchain.com/v0.1/docs/expression_language/interface/), the basic building block of the LangChain Expression Language (LCEL). This means they support `invoke`, `ainvoke`, `stream`, `astream`, `batch`, `abatch`, `astream_log` calls.

`PromptTemplate` accepts a dictionary (of the prompt variables) and returns a `StringPromptValue`. A `ChatPromptTemplate` accepts a dictionary and returns a `ChatPromptValue`.

In [ ]:
text_prompts = ["what is your name?", "explain commercial real estate to me"]
chat_prompts = [chat_template.invoke({'user_prompt' : prompt}) for prompt in text_prompts]
chat_prompts

In [ ]:
chat_prompts[1]

In [ ]:
print(chat_prompts[1].to_string())

In [ ]:
chat_prompts[1].to_messages()

In [ ]:
responses = chatgpt.map().invoke(chat_prompts)
for response in responses:
  print(response.content)
  print('-----')